In [ ]:
# ==============================================================================
# NOTEBOOK 99 — AUDITORIA DOS ARTEFATOS PAS/UnB
#
# CÉLULA 1 — CONFIGURAÇÃO
#
# Responsabilidades
#   • Importações
#   • Configuração dos diretórios do projeto
#   • Carregamento dos catálogos
# ==============================================================================

from pathlib import Path
import json

import pandas as pd
import numpy as np

from IPython.display import display

# ------------------------------------------------------------------------------
# Diretórios do projeto
# ------------------------------------------------------------------------------


def resolver_base() -> Path:
    """Resolve o diretório raiz do projeto em qualquer ambiente."""
    cwd = Path.cwd().resolve()

    for candidate in [cwd, *cwd.parents]:
        if (candidate / "tsv").exists() and (candidate / "catalogos").exists():
            return candidate

    return cwd


BASE = resolver_base()

DIR_CATALOGOS = BASE / "catalogos"
DIR_DIAGNOSTICOS = BASE / "diagnosticos"

ARQUIVO_RESUMO = DIR_DIAGNOSTICOS / "resumo_processamento.csv"

# ------------------------------------------------------------------------------
# Catálogos
# ------------------------------------------------------------------------------

def carregar_json(arquivo):

    with open(arquivo, encoding="utf-8") as f:
        return json.load(f)

INDICE_CURSOS = carregar_json(
    DIR_CATALOGOS / "indice_cursos.json"
)

CATALOGO_CURSOS = carregar_json(
    DIR_CATALOGOS / "catalogo_cursos.json"
)

# ------------------------------------------------------------------------------
# Status das auditorias
# ------------------------------------------------------------------------------

STATUS_OK = "OK"
STATUS_AVISO = "AVISO"
STATUS_ERRO = "ERRO"
STATUS_FALHA = "FALHA"

# ------------------------------------------------------------------------------
# Inicialização
# ------------------------------------------------------------------------------

print("=" * 80)
print("NOTEBOOK 99 — AUDITORIA DOS ARTEFATOS PAS/UnB")
print("=" * 80)

print(f"BASE................. {BASE}")
print(f"DIAGNÓSTICOS......... {DIR_DIAGNOSTICOS}")
print(f"CATÁLOGOS............ {DIR_CATALOGOS}")

print(f"Cursos canônicos..... {len(CATALOGO_CURSOS)}")
print(f"Variantes............ {len(INDICE_CURSOS)}")

Mounted at /content/drive
NOTEBOOK 99 — AUDITORIA DOS ARTEFATOS PAS/UnB
BASE................. /content/drive/MyDrive/Dados/PAS
DIAGNÓSTICOS......... /content/drive/MyDrive/Dados/PAS/diagnosticos
CATÁLOGOS............ /content/drive/MyDrive/Dados/PAS/catalogos
Cursos canônicos..... 90
Variantes............ 132


In [2]:
# ==============================================================================
# NOTEBOOK 99 — AUDITORIA DOS ARTEFATOS PAS/UnB
#
# CÉLULA 2 — CARREGAMENTO DOS ARTEFATOS
#
# Responsabilidades
#   • Carregar o resumo do processamento
#   • Descobrir os triênios disponíveis
#   • Carregar todos os CSVs e JSONs
#   • Construir a estrutura ARTEFATOS
# ==============================================================================

# ------------------------------------------------------------------------------
# Verificação do diretório
# ------------------------------------------------------------------------------

if not DIR_DIAGNOSTICOS.exists():

    raise FileNotFoundError(
        f"Diretório não encontrado:\n{DIR_DIAGNOSTICOS}"
    )

# ------------------------------------------------------------------------------
# Resumo do processamento
# ------------------------------------------------------------------------------

if not ARQUIVO_RESUMO.exists():

    raise FileNotFoundError(
        f"Arquivo não encontrado:\n{ARQUIVO_RESUMO}"
    )

RESUMO_PROCESSAMENTO = pd.read_csv(ARQUIVO_RESUMO)

print(f"Resumo carregado: {len(RESUMO_PROCESSAMENTO)} triênio(s).")

# ------------------------------------------------------------------------------
# Triênios disponíveis
# ------------------------------------------------------------------------------

TRIENIOS = sorted(
    RESUMO_PROCESSAMENTO["Triênio"].unique().tolist()
)

# ------------------------------------------------------------------------------
# Carregamento dos artefatos
# ------------------------------------------------------------------------------

ARTEFATOS = {}

for trienio in TRIENIOS:

    arquivo_dados = DIR_DIAGNOSTICOS / f"{trienio}.csv"

    arquivo_auditoria = DIR_DIAGNOSTICOS / f"{trienio}_auditoria.csv"

    arquivo_json = DIR_DIAGNOSTICOS / f"{trienio}_resumo.json"

    if not arquivo_dados.exists():
        raise FileNotFoundError(arquivo_dados)

    if not arquivo_auditoria.exists():
        raise FileNotFoundError(arquivo_auditoria)

    if not arquivo_json.exists():
        raise FileNotFoundError(arquivo_json)

    df = pd.read_csv(arquivo_dados)

    df_auditoria = pd.read_csv(arquivo_auditoria)

    with open(arquivo_json, encoding="utf-8") as f:
        dados = json.load(f)

    resumo = (
        RESUMO_PROCESSAMENTO
        .loc[RESUMO_PROCESSAMENTO["Triênio"] == trienio]
        .iloc[0]
    )

    ARTEFATOS[trienio] = {

        "csv": df,

        "auditoria": df_auditoria,

        "json": dados,

        "resumo": resumo,

        "auditorias": {},

        "parecer": None

    }

# ------------------------------------------------------------------------------
# Resumo
# ------------------------------------------------------------------------------

print()

print("=" * 80)
print("ARTEFATOS CARREGADOS")
print("=" * 80)

for trienio in TRIENIOS:

    n = len(ARTEFATOS[trienio]["csv"])
    n_aud = len(ARTEFATOS[trienio]["auditoria"])

    print(
        f"{trienio:12} "
        f"{n:5} registros | "
        f"{n_aud:5} auditoria"
    )

print()

print(f"Triênios carregados: {len(TRIENIOS)}")


Resumo carregado: 6 triênio(s).

ARTEFATOS CARREGADOS
2018-2020      436 registros |   436 auditoria
2019-2021      463 registros |   463 auditoria
2020-2022      449 registros |   449 auditoria
2021-2023      445 registros |   445 auditoria
2022-2024      168 registros |   168 auditoria
2023-2025      345 registros |   345 auditoria

Triênios carregados: 6


In [3]:
# =============================================================================
# NOTEBOOK 99 — AUDITORIA DOS ARTEFATOS PAS/UnB
#
# CÉLULA 3 — FUNÇÕES AUXILIARES
#
# Responsabilidades
#   • Configuração da auditoria
#   • Estrutura padronizada dos resultados
#   • Funções utilitárias
# =============================================================================

# -----------------------------------------------------------------------------
# Configuração da auditoria
# -----------------------------------------------------------------------------

COLUNAS_OBRIGATORIAS = [

    "Curso",
    "Campus",
    "Modalidade",
    "Turno"

]

CHAVE_REGISTRO = [

    "Curso",
    "Campus",
    "Modalidade",
    "Turno"

]

VALORES_INVALIDOS = {

    "",
    " ",
    "N/A",
    "NA",
    "NULL",
    "None",
    "---",
    "?"

}

# -----------------------------------------------------------------------------
# Estrutura padrão de retorno
# -----------------------------------------------------------------------------

def criar_resultado(nome):
    """
    Cria a estrutura padrão utilizada por todas
    as auditorias.
    """

    return {

        "nome": nome,

        "status": STATUS_OK,

        "mensagem": "",

        "n_erros": 0,

        "erros": pd.DataFrame(),

        "estatisticas": {}

    }


def finalizar_resultado(
    resultado,
    status,
    mensagem="",
    erros=None,
    estatisticas=None
):
    """
    Finaliza o resultado de uma auditoria.
    """

    resultado["status"] = status
    resultado["mensagem"] = mensagem

    if erros is None:
        erros = pd.DataFrame()

    resultado["erros"] = erros
    resultado["n_erros"] = len(erros)

    if estatisticas is not None:
        resultado["estatisticas"] = estatisticas

    return resultado


# -----------------------------------------------------------------------------
# Utilidades
# -----------------------------------------------------------------------------

def dataframe_json(dados):
    """
    Converte o JSON exportado para DataFrame.
    """

    return pd.DataFrame(dados)


def colunas_ausentes(df):
    """
    Retorna as colunas obrigatórias ausentes.
    """

    return sorted(
        set(COLUNAS_OBRIGATORIAS) -
        set(df.columns)
    )


def valores_unicos(df, coluna):
    """
    Valores distintos de uma coluna.
    """

    return sorted(
        df[coluna]
        .dropna()
        .unique()
    )


def comparar_conjuntos(observado, esperado):
    """
    Retorna elementos presentes em 'observado'
    mas ausentes em 'esperado'.
    """

    return sorted(
        set(observado) -
        set(esperado)
    )


def imprimir_status(resultado):
    """
    Exibe o resultado resumido de uma auditoria.
    """

    print(
        f"[{resultado['status']}] "
        f"{resultado['nome']}"
    )

    if resultado["mensagem"]:
        print(f"    {resultado['mensagem']}")

    if resultado["n_erros"]:

        print(
            f"    Ocorrências: "
            f"{resultado['n_erros']}"
        )

In [4]:
# ==============================================================================
# NOTEBOOK 99 — AUDITORIA DOS ARTEFATOS PAS/UnB
#
# CÉLULA 4 — AUDITORIAS ESTRUTURAIS
#
# Responsabilidades
#   • Verificar a estrutura do CSV
#   • Verificar a estrutura do JSON
#   • Validar a consistência entre CSV e resumo JSON
# ==============================================================================


# ------------------------------------------------------------------------------
# Estrutura do CSV
# ------------------------------------------------------------------------------

def auditar_csv(trienio):

    resultado = criar_resultado("Estrutura do CSV")

    df = ARTEFATOS[trienio]["csv"]

    erros = []

    if df.empty:

        erros.append({
            "Problema": "CSV vazio"
        })

    faltantes = colunas_ausentes(df)

    for coluna in faltantes:

        erros.append({

            "Problema": "Coluna obrigatória ausente",

            "Coluna": coluna

        })

    if erros:

        return finalizar_resultado(

            resultado,

            STATUS_ERRO,

            "Problemas estruturais encontrados.",

            pd.DataFrame(erros)

        )

    return finalizar_resultado(

        resultado,

        STATUS_OK,

        estatisticas={

            "Registros": len(df),

            "Colunas": len(df.columns)

        }

    )


# ------------------------------------------------------------------------------
# Estrutura do JSON
# ------------------------------------------------------------------------------

def auditar_json(trienio):

    resultado = criar_resultado("Estrutura do JSON")

    resumo = ARTEFATOS[trienio]["json"]

    chaves_obrigatorias = [

        "subprograma",

        "shape",

        "registros",

        "campi",

        "cursos",

        "modalidades",

        "nota_min_global",

        "nota_max_global",

        "campi_contagem",

        "modalidades_contagem"

    ]

    faltantes = [

        chave

        for chave in chaves_obrigatorias

        if chave not in resumo

    ]

    if faltantes:

        return finalizar_resultado(

            resultado,

            STATUS_ERRO,

            "JSON incompleto.",

            pd.DataFrame({

                "Chave ausente": faltantes

            })

        )

    return finalizar_resultado(resultado, STATUS_OK)


# ------------------------------------------------------------------------------
# Consistência CSV × JSON
# ------------------------------------------------------------------------------

def auditar_consistencia(trienio):

    resultado = criar_resultado("Consistência CSV × JSON")

    df = ARTEFATOS[trienio]["csv"]

    resumo = ARTEFATOS[trienio]["json"]

    erros = []

    verificacoes = [

        ("registros", len(df)),

        ("campi", df["Campus"].nunique()),

        ("cursos", df["Curso"].nunique()),

        ("modalidades", df["Modalidade"].nunique()),

        ("nota_min_global", df["Nota Mínima"].min()),

        ("nota_max_global", df["Nota Máxima"].max())

    ]

    for chave, observado in verificacoes:

        esperado = resumo[chave]

        if observado != esperado:

            erros.append({

                "Campo": chave,

                "CSV": observado,

                "JSON": esperado

            })

    if erros:

        return finalizar_resultado(

            resultado,

            STATUS_ERRO,

            "Resumo JSON inconsistente com o CSV.",

            pd.DataFrame(erros)

        )

    return finalizar_resultado(resultado, STATUS_OK)


# ------------------------------------------------------------------------------
# Execução
# ------------------------------------------------------------------------------

def executar_auditorias_estruturais(trienio):

    return [

        auditar_csv(trienio),

        auditar_json(trienio),

        auditar_consistencia(trienio)

    ]

In [5]:
from pathlib import Path

for arquivo in sorted(DIR_DIAGNOSTICOS.glob("*")):
    print(arquivo.name)

2018-2020.csv
2018-2020_auditoria.csv
2018-2020_resumo.json
2019-2021.csv
2019-2021_auditoria.csv
2019-2021_resumo.json
2020-2022.csv
2020-2022_auditoria.csv
2020-2022_resumo.json
2021-2023.csv
2021-2023_auditoria.csv
2021-2023_resumo.json
2022-2024.csv
2022-2024_auditoria.csv
2022-2024_resumo.json
2023-2025.csv
2023-2025_auditoria.csv
2023-2025_resumo.json
resumo_processamento.csv


In [6]:
# ==============================================================================
# NOTEBOOK 99 — AUDITORIA DOS ARTEFATOS PAS/UnB
#
# CÉLULA 5 — AUDITORIAS DE DOMÍNIO
#
# Responsabilidades
#   • Validar cursos
#   • Validar campi
#   • Validar modalidades
#   • Validar turnos
# ==============================================================================


# ------------------------------------------------------------------------------
# Cursos
# ------------------------------------------------------------------------------

def auditar_cursos(trienio):

    resultado = criar_resultado("Cursos")

    df = ARTEFATOS[trienio]["csv"]

    encontrados = sorted(df["Curso"].dropna().unique())

    invalidos = sorted(
        curso
        for curso in encontrados
        if curso not in CATALOGO_CURSOS
    )

    if invalidos:

        return finalizar_resultado(

            resultado,

            STATUS_ERRO,

            "Cursos inexistentes no catálogo.",

            pd.DataFrame({"Curso": invalidos})

        )

    return finalizar_resultado(

        resultado,

        STATUS_OK,

        estatisticas={

            "Cursos": len(encontrados)

        }

    )


# ------------------------------------------------------------------------------
# Campi
# ------------------------------------------------------------------------------

def auditar_campi(trienio):

    resultado = criar_resultado("Campi")

    df = ARTEFATOS[trienio]["csv"]

    campi = sorted(df["Campus"].dropna().unique())

    return finalizar_resultado(

        resultado,

        STATUS_OK,

        estatisticas={

            "Campi": len(campi)

        }

    )


# ------------------------------------------------------------------------------
# Turnos
# ------------------------------------------------------------------------------

def auditar_turnos(trienio):

    resultado = criar_resultado("Turnos")

    df = ARTEFATOS[trienio]["csv"]

    turnos = sorted(df["Turno"].dropna().unique())

    return finalizar_resultado(

        resultado,

        STATUS_OK,

        estatisticas={

            "Turnos": len(turnos)

        }

    )


# ------------------------------------------------------------------------------
# Modalidades
# ------------------------------------------------------------------------------

def auditar_modalidades(trienio):

    resultado = criar_resultado("Modalidades")

    df = ARTEFATOS[trienio]["csv"]

    modalidades = sorted(df["Modalidade"].dropna().unique())

    return finalizar_resultado(

        resultado,

        STATUS_OK,

        estatisticas={

            "Modalidades": len(modalidades)

        }

    )


# ------------------------------------------------------------------------------
# Combinações Curso × Campus × Turno
# ------------------------------------------------------------------------------

def auditar_combinacoes(trienio):

    resultado = criar_resultado("Combinações")

    df = ARTEFATOS[trienio]["csv"]

    combinacoes = (
        df[
            [
                "Curso",
                "Campus",
                "Turno"
            ]
        ]
        .drop_duplicates()
    )

    return finalizar_resultado(

        resultado,

        STATUS_OK,

        estatisticas={

            "Combinações": len(combinacoes)

        }

    )


# ------------------------------------------------------------------------------
# Execução
# ------------------------------------------------------------------------------

def executar_auditorias_dominio(trienio):

    return [

        auditar_cursos(trienio),

        auditar_campi(trienio),

        auditar_turnos(trienio),

        auditar_modalidades(trienio),

        auditar_combinacoes(trienio),

    ]

In [7]:
# ==============================================================================
# NOTEBOOK 99 — AUDITORIA DOS ARTEFATOS PAS/UnB
#
# CÉLULA 6 — AUDITORIAS DE INTEGRIDADE
#
# Responsabilidades
#   • Detectar registros duplicados
#   • Verificar valores ausentes
#   • Validar intervalo das notas
# ==============================================================================

# ------------------------------------------------------------------------------
# Duplicidades
# ------------------------------------------------------------------------------

def auditar_duplicidades(trienio):

    resultado = criar_resultado("Duplicidades")

    df = ARTEFATOS[trienio]["csv"]

    duplicados = df.duplicated(
        subset=CHAVE_REGISTRO,
        keep=False
    )

    if duplicados.any():

        erros = (
            df.loc[duplicados]
            .sort_values(CHAVE_REGISTRO)
            .reset_index(drop=True)
        )

        return finalizar_resultado(

            resultado,

            STATUS_ERRO,

            "Registros duplicados encontrados.",

            erros

        )

    return finalizar_resultado(resultado, STATUS_OK)


# ------------------------------------------------------------------------------
# Valores ausentes
# ------------------------------------------------------------------------------

def auditar_campos_obrigatorios(trienio):

    resultado = criar_resultado("Campos obrigatórios")

    df = ARTEFATOS[trienio]["csv"]

    erros = []

    for coluna in COLUNAS_OBRIGATORIAS:

        quantidade = int(df[coluna].isna().sum())

        if quantidade > 0:

            erros.append({

                "Campo": coluna,

                "Ocorrências": quantidade

            })

    if erros:

        return finalizar_resultado(

            resultado,

            STATUS_ERRO,

            "Existem valores ausentes.",

            pd.DataFrame(erros)

        )

    return finalizar_resultado(resultado, STATUS_OK)


# ------------------------------------------------------------------------------
# Intervalo das notas
# ------------------------------------------------------------------------------

def auditar_notas(trienio):

    resultado = criar_resultado("Notas")

    df = ARTEFATOS[trienio]["csv"]

    erros = df[
        df["Nota Mínima"] > df["Nota Máxima"]
    ]

    if not erros.empty:

        return finalizar_resultado(

            resultado,

            STATUS_ERRO,

            "Existem registros com Nota Mínima maior que Nota Máxima.",

            erros

        )

    return finalizar_resultado(resultado, STATUS_OK)


# ------------------------------------------------------------------------------
# Execução
# ------------------------------------------------------------------------------

def executar_auditorias_integridade(trienio):

    return [

        auditar_duplicidades(trienio),

        auditar_campos_obrigatorios(trienio),

        auditar_notas(trienio),

    ]

In [8]:
# ==============================================================================
# NOTEBOOK 99 — AUDITORIA DOS ARTEFATOS PAS/UnB
#
# CÉLULA 7 — EXECUÇÃO E RELATÓRIO FINAL
#
# Responsabilidades
#   • Executar todas as auditorias
#   • Consolidar os resultados
#   • Emitir parecer por triênio
#   • Gerar relatório final
# ==============================================================================

# ------------------------------------------------------------------------------
# Grupos de auditorias
# ------------------------------------------------------------------------------

AUDITORIAS = [

    executar_auditorias_estruturais,

    executar_auditorias_dominio,

    executar_auditorias_integridade,

]

# ------------------------------------------------------------------------------
# Parecer
# ------------------------------------------------------------------------------

def emitir_parecer(resultados):

    status = [r["status"] for r in resultados]

    if STATUS_FALHA in status:
        return STATUS_FALHA

    if STATUS_ERRO in status:
        return STATUS_ERRO

    if STATUS_AVISO in status:
        return STATUS_AVISO

    return STATUS_OK


# ------------------------------------------------------------------------------
# Auditoria de um triênio
# ------------------------------------------------------------------------------

def auditar_trienio(trienio):

    print()
    print("=" * 80)
    print(trienio)
    print("=" * 80)

    resultados = []

    estatisticas = {}

    for grupo in AUDITORIAS:

        auditorias = grupo(trienio)

        resultados.extend(auditorias)

    for resultado in resultados:

        imprimir_status(resultado)

        if resultado["estatisticas"]:

            estatisticas.update(resultado["estatisticas"])

    parecer = emitir_parecer(resultados)

    ARTEFATOS[trienio]["auditorias"] = {

        r["nome"]: r

        for r in resultados

    }

    ARTEFATOS[trienio]["parecer"] = parecer

    ARTEFATOS[trienio]["estatisticas"] = estatisticas

    print()

    print(f"PARECER FINAL: {parecer}")

    return {

        "Triênio": trienio,

        "Parecer": parecer,

        "Erros": sum(r["n_erros"] for r in resultados),

        **estatisticas

    }


# ------------------------------------------------------------------------------
# Auditoria completa
# ------------------------------------------------------------------------------

def auditar_todos():

    linhas = []

    for trienio in TRIENIOS:

        linhas.append(

            auditar_trienio(trienio)

        )

    return pd.DataFrame(linhas)


# ------------------------------------------------------------------------------
# Execução
# ------------------------------------------------------------------------------

print("=" * 80)
print("AUDITORIA DOS ARTEFATOS")
print("=" * 80)

RELATORIO = auditar_todos()

print()

print("=" * 80)
print("RESUMO FINAL")
print("=" * 80)

display(RELATORIO)

print()

print("=" * 80)
print("AUDITORIA CONCLUÍDA")
print("=" * 80)

AUDITORIA DOS ARTEFATOS

2018-2020
[OK] Estrutura do CSV
[OK] Estrutura do JSON
[OK] Consistência CSV × JSON
[OK] Cursos
[OK] Campi
[OK] Turnos
[OK] Modalidades
[OK] Combinações
[OK] Duplicidades
[OK] Campos obrigatórios
[OK] Notas

PARECER FINAL: OK

2019-2021
[OK] Estrutura do CSV
[OK] Estrutura do JSON
[OK] Consistência CSV × JSON
[OK] Cursos
[OK] Campi
[OK] Turnos
[OK] Modalidades
[OK] Combinações
[OK] Duplicidades
[OK] Campos obrigatórios
[OK] Notas

PARECER FINAL: OK

2020-2022
[OK] Estrutura do CSV
[OK] Estrutura do JSON
[OK] Consistência CSV × JSON
[OK] Cursos
[OK] Campi
[OK] Turnos
[OK] Modalidades
[OK] Combinações
[OK] Duplicidades
[OK] Campos obrigatórios
[OK] Notas

PARECER FINAL: OK

2021-2023
[OK] Estrutura do CSV
[OK] Estrutura do JSON
[OK] Consistência CSV × JSON
[OK] Cursos
[OK] Campi
[OK] Turnos
[OK] Modalidades
[OK] Combinações
[OK] Duplicidades
[OK] Campos obrigatórios
[OK] Notas

PARECER FINAL: OK

2022-2024
[OK] Estrutura do CSV
[OK] Estrutura do JSON
[OK] Consist

,Triênio,Parecer,Erros,Registros,Colunas,Cursos,Campi,Turnos,Modalidades,Combinações
0,2018-2020,OK,0,436,14,88,4,2,8,100
1,2019-2021,OK,0,463,14,85,4,2,8,99
2,2020-2022,OK,0,449,14,88,4,2,8,99
3,2021-2023,OK,0,445,14,86,4,2,9,98
4,2022-2024,OK,0,168,14,48,3,2,7,54
5,2023-2025,OK,0,345,14,85,4,2,9,96



AUDITORIA CONCLUÍDA
